In [ ]:
import gdown

file_id = '1ewhkWzZH-uYm1Znn2bo3Z5u1K2xWbhsl'
url = f'https://drive.google.com/uc?id={file_id}'

gdown.download(url, quiet=False)

In [ ]:
%pip install -r requirements.txt

In [ ]:
import zipfile
from pathlib import Path

zip_path = "processed.zip"
out_dir = "./data/processed"

out_dir = Path(out_dir)
out_dir.mkdir(parents=True, exist_ok=True)

print(f"Extracting {zip_path} -> {out_dir} ...")
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(out_dir)
print("Done.")

## Dataset

In [ ]:
import json
from functools import lru_cache
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import Dataset


class QMugsDataset(Dataset):
    def __init__(
        self,
        processed_path,
        split: str = "train",
        split_ratio: tuple = (0.8, 0.1, 0.1),
        seed: int = 42,
        cache_chunks: int = 2,
        max_samples: int = None,
    ):
        processed_path = Path(processed_path)
        self._processed_path = processed_path

        with open(processed_path / "meta.json") as f:
            meta = json.load(f)

        self._chunk_names = meta["chunk_files"]
        chunk_sizes = meta["chunk_sizes"]

        self._chunk_offsets = np.cumsum([0] + chunk_sizes[:-1])
        self._total = sum(chunk_sizes)

        rng = np.random.default_rng(seed)
        perm = rng.permutation(self._total)

        train_end = int(self._total * split_ratio[0])
        val_end = train_end + int(self._total * split_ratio[1])

        splits = {
            "train": perm[:train_end],
            "val":   perm[train_end:val_end],
            "test":  perm[val_end:],
        }
        if split not in splits:
            raise ValueError(f"split must be 'train', 'val', or 'test'; got '{split}'")
        split_indices = splits[split]

        # 청크 순서 셔플 + 청크 내 셔플 → DataLoader shuffle=False로 cache hit 보장
        chunk_of = np.searchsorted(self._chunk_offsets, split_indices, side="right") - 1
        chunk_order = rng.permutation(len(self._chunk_names))
        ordered = []
        for c in chunk_order:
            mask = chunk_of == c
            if mask.any():
                items = split_indices[mask].copy()
                rng.shuffle(items)
                ordered.append(items)
        self.indices = np.concatenate(ordered) if ordered else split_indices
        if max_samples is not None:
            self.indices = self.indices[:max_samples]

        with open(processed_path / "stats.json") as f:
            stats = json.load(f)
        prop_cols = list(stats.keys())
        self.mean = torch.tensor([stats[c]["mean"] for c in prop_cols], dtype=torch.float)
        self.std  = torch.tensor([stats[c]["std"]  for c in prop_cols], dtype=torch.float)

        self._load_chunk = lru_cache(maxsize=cache_chunks)(self._load_chunk_uncached)

    def _load_chunk_uncached(self, chunk_idx: int) -> list:
        return torch.load(
            self._processed_path / self._chunk_names[chunk_idx],
            weights_only=False,
        )

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, idx: int):
        global_idx = int(self.indices[idx])
        chunk_idx = int(np.searchsorted(self._chunk_offsets, global_idx, side="right") - 1)
        local_idx = global_idx - int(self._chunk_offsets[chunk_idx])
        data = self._load_chunk(chunk_idx)[local_idx].clone()
        data.p = (data.p_raw - self.mean) / self.std
        return data

## Model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINEConv, VGAE, global_mean_pool
from torch_geometric.utils import to_dense_adj

# C, H, O, N, S, P, F, Cl, Br, I
ELEM_ATOMIC_NUMS = [6, 1, 8, 7, 16, 15, 9, 17, 35, 53]

LATENT_DIM = 128
HIDDEN_DIM = 256
ATOM_EMB_DIM = 64
COND_DIM = 128
ENCODER_LAYERS = 4
DECODER_LAYERS = 4
N_BOND_TYPES = 4 # 1중 / 2중 / 3중 / 컨쥬게이션

# atom_embedding + charge + chirality
NODE_DIM = ATOM_EMB_DIM + 1 + 3
# bond_type + bond_stereo + dihedral
EDGE_DIM = 4 + 5 + 1


def mlp(dims, dropout = 0.0):
    layers = []
    for i in range(len(dims) - 1):
        layers.append(nn.Linear(dims[i], dims[i+1]))
        if i < len(dims) - 2:
            if dropout:
                layers += [nn.ReLU(), nn.Dropout(dropout)]
            else:
                layers += [nn.ReLU()]
    return nn.Sequential(*layers)

def to_atom_tensors(a: torch.Tensor, device: torch.device):
    atom_lists = []
    for row in a:
        atoms = []
        for atomic_num, cnt in zip(ELEM_ATOMIC_NUMS, row[:10].long().tolist()):
            atoms.extend([atomic_num] * cnt)
        atom_lists.append(atoms)

    N_max = max(len(al) for al in atom_lists)
    B = len(atom_lists)

    atom_idx = torch.zeros(B, N_max, dtype=torch.long, device=device)
    mask = torch.zeros(B, N_max, dtype=torch.bool, device=device)
    for b, atoms in enumerate(atom_lists):
        n = len(atoms)
        atom_idx[b, :n] = torch.tensor(atoms, dtype=torch.long, device=device)
        mask[b, :n] = True

    return atom_idx, mask


class GraphEncoder(nn.Module):

    def __init__(self):
        super().__init__()
        self.atom_emb = nn.Embedding(120, ATOM_EMB_DIM)
        self.atom_proj = mlp([NODE_DIM, HIDDEN_DIM])

        self.convs = nn.ModuleList([
            GINEConv(
                mlp([HIDDEN_DIM, HIDDEN_DIM, HIDDEN_DIM]),
                edge_dim=EDGE_DIM,
            )
            for _ in range(ENCODER_LAYERS)
        ])
        self.norms = nn.ModuleList([nn.LayerNorm(HIDDEN_DIM) for _ in range(ENCODER_LAYERS)])

        self.pool_proj = mlp([HIDDEN_DIM + 5 + 11, HIDDEN_DIM])

    def forward(self, batch):
        h = torch.cat(
            [self.atom_emb(batch.z), batch.charge.unsqueeze(-1), batch.chirality],
            dim=-1,
        )
        h = self.atom_proj(h)

        edge_attr = torch.cat(
            [batch.bond_type, batch.bond_stereo, batch.dihedral.unsqueeze(-1)],
            dim=-1,
        )

        for conv, norm in zip(self.convs, self.norms):
            h = norm(F.relu(conv(h, batch.edge_index, edge_attr)))

        g = global_mean_pool(h, batch.batch)
        B = g.size(0)

        return self.pool_proj(torch.cat([g, batch.p.view(B, -1), batch.a.view(B, -1)], dim=-1))
    

class MoleculeDecoder(nn.Module):

    def __init__(self):
        super().__init__()
        self.atom_emb = nn.Embedding(120, ATOM_EMB_DIM)
        self.cond_proj = mlp([LATENT_DIM + 5 + 11, COND_DIM])
        self.init_proj = mlp([ATOM_EMB_DIM + COND_DIM, HIDDEN_DIM])

        self.attn_layers = nn.ModuleList([
            nn.MultiheadAttention(HIDDEN_DIM, num_heads=8, batch_first=True)
            for _ in range(DECODER_LAYERS)
        ])
        self.attn_norms = nn.ModuleList([nn.LayerNorm(HIDDEN_DIM) for _ in range(DECODER_LAYERS)])
        self.ffns = nn.ModuleList([
            mlp([HIDDEN_DIM, HIDDEN_DIM * 2, HIDDEN_DIM])
            for _ in range(DECODER_LAYERS)
        ])
        self.ffn_norms = nn.ModuleList([nn.LayerNorm(HIDDEN_DIM) for _ in range(DECODER_LAYERS)])

        pair_dim = HIDDEN_DIM * 2
        self.bond_exist = mlp([pair_dim, HIDDEN_DIM, 1])
        self.bond_type = mlp([pair_dim, HIDDEN_DIM, N_BOND_TYPES])

    def forward(self, z, p, a):
        device = z.device

        c = self.cond_proj(torch.cat([z, p, a], dim=-1)) # [B, COND_DIM]

        atom_idx, atom_mask = to_atom_tensors(a, device) # [B, N_max]
        N_max = atom_idx.size(1)

        h_atom = self.atom_emb(atom_idx) # [B, N_max, ATOM_EMB_DIM]
        c_exp = c.unsqueeze(1).expand(-1, N_max, -1) # [B, N_max, COND_DIM]
        h = self.init_proj(torch.cat([h_atom, c_exp], dim=-1)) # [B, N_max, HIDDEN_DIM]

        # atom_mask : True(원자), False(패딩)
        # key_pad_mask (MultiHeadAttention) : True(패딩), False(원자)
        key_pad_mask = ~atom_mask
        for attn, a_norm, ffn, f_norm in zip(
            self.attn_layers, self.attn_norms, self.ffns, self.ffn_norms):
            h2, _ = attn(h, h, h, key_padding_mask=key_pad_mask)
            h = a_norm(h + h2)
            h = f_norm(h + ffn(h))

        hi = h.unsqueeze(2).expand(-1, -1, N_max, -1) # [B, N_max, N_max, H]
        hj = h.unsqueeze(1).expand(-1, N_max, -1, -1)
        pair = torch.cat([hi + hj, hi * hj], dim=-1)

        return self.bond_exist(pair).squeeze(-1), self.bond_type(pair), atom_mask
    

def reconstruction_loss(
    bond_exist_logit,
    bond_type_logit,
    atom_mask,
    batch
):
    device = bond_exist_logit.device
    B, N_max, _ = bond_exist_logit.shape

    true_adj = to_dense_adj(batch.edge_index, batch=batch.batch, max_num_nodes=N_max)

    # (B, N_max, N_max)
    bond_class = batch.bond_type.argmax(dim=-1).float()
    true_bond_type = to_dense_adj(
        batch.edge_index, batch=batch.batch,
        edge_attr=bond_class.unsqueeze(-1),
        max_num_nodes=N_max,
    ).squeeze(-1).long()

    tri = torch.triu(torch.ones(N_max, N_max, dtype=torch.bool, device=device), diagonal=1)
    pair_mask = atom_mask.unsqueeze(2) & atom_mask.unsqueeze(1) & tri # [B, N_max, N_max]

    n_pairs = pair_mask.sum().float()
    n_pos = (true_adj * pair_mask).sum().float().clamp(min=1)
    pos_weight = ((n_pairs - n_pos) / n_pos).clamp(max=50)

    exist_loss = F.binary_cross_entropy_with_logits(
        bond_exist_logit[pair_mask],
        true_adj[pair_mask],
        pos_weight=pos_weight,
    )

    pos_mask = pair_mask & (true_adj > 0.5)
    type_loss = (
        F.cross_entropy(bond_type_logit[pos_mask], true_bond_type[pos_mask])
        if pos_mask.any()
        else bond_exist_logit.new_tensor(0.0)
    )

    return exist_loss + type_loss, exist_loss, type_loss


def free_bits_kl(mu, logstd, lambda_=0.1):
    # 지원별 KL을 lambda_ 이상으로 강제해 posterior collapse 방지
    kl_per_dim = -0.5 * (1 + 2 * logstd - mu.pow(2) - (2 * logstd).exp())
    return kl_per_dim.clamp(min=lambda_).sum(dim=-1).mean()

class MoleculeCVAE(VGAE):

    def __init__(self, beta = 1.0):
        super().__init__(encoder=GraphEncoder())
        self.mu_head = nn.Linear(HIDDEN_DIM, LATENT_DIM)
        self.logstd_head = nn.Linear(HIDDEN_DIM, LATENT_DIM)
        self.molecule_decoder = MoleculeDecoder()
        self.beta = beta

    def encode(self, batch):
        h = self.encoder(batch)
        self.__mu__ = self.mu_head(h)
        self.__logstd__ = self.logstd_head(h).clamp(max=10.0)
        return self.reparametrize(self.__mu__, self.__logstd__)

    def forward(self, batch):
        h = self.encoder(batch)
        self.__mu__ = self.mu_head(h)
        self.__logstd__ = self.logstd_head(h).clamp(max=10.0)
        # warmup(beta=0) 동안은 reparameterization 노이즈 없이 mu를 직접 사용
        z = self.__mu__ if self.beta == 0 else self.reparametrize(self.__mu__, self.__logstd__)
        B = z.size(0)
        p = batch.p.view(B, -1)
        a = batch.a.view(B, -1)

        exist_logit, type_logit, atom_mask = self.molecule_decoder(z, p, a)
        r_loss, e_loss, t_loss = reconstruction_loss(exist_logit, type_logit, atom_mask, batch)
        kl = free_bits_kl(self.__mu__, self.__logstd__)

        return {
            "loss": r_loss + self.beta * kl,
            "reconstruction_loss": r_loss,
            "exist_loss": e_loss,
            "type_loss": t_loss,
            "kl_loss": kl
        }

    @torch.no_grad()
    def generate(self, p, a, threshold = 0.5):
        self.eval()
        B = p.size(0)
        z = torch.randn(B, LATENT_DIM, device=p.device)

        exist_logit, type_logit, atom_mask = self.molecule_decoder(z, p, a)
        bond_prob = exist_logit.sigmoid()

        results = []
        for b in range(B):
            n = atom_mask[b].sum().item()
            prob = bond_prob[b, :n, :n]
            btype = type_logit[b, :n, :n].argmax(dim=-1)

            tri = torch.triu(torch.ones(n, n, dtype=torch.bool, device=p.device), diagonal=1)
            edge = (prob > threshold) & tri

            u, v = edge.nonzero(as_tuple=True)
            edge_index = torch.stack([torch.cat([u, v]), torch.cat([v, u])])
            bond_types = torch.cat([btype[u, v], btype[v, u]])

            results.append({
                "edge_index": edge_index,
                "bond_types": bond_types,
                "n_atoms": n
            })

        return results

## Train

In [ ]:
import math
import time
from pathlib import Path
from tqdm.notebook import tqdm
from types import SimpleNamespace

import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch_geometric.loader import DataLoader


def cyclical_beta(step, total_steps, n_cycles=4, max_beta=1.0, ratio=0.5, warmup_steps=0):
    if step < warmup_steps:
        return 0.0
    effective_step = step - warmup_steps
    effective_total = total_steps - warmup_steps
    cycle_steps = effective_total / n_cycles
    cycle_pos = effective_step % cycle_steps
    anneal_steps = cycle_steps * ratio
    if cycle_pos < anneal_steps:
        t = cycle_pos / anneal_steps
        return max_beta * 0.5 * (1 - math.cos(math.pi * t))
    return max_beta


def run_epoch(model, loader, optimizer, beta, device, train, epoch):
    model.train(train)
    totals = {
        "loss": 0,
        "reconstruction_loss": 0,
        "exist_loss": 0,
        "type_loss": 0,
        "kl_loss": 0
    }

    with torch.set_grad_enabled(train):
        for batch in tqdm(loader, desc=f"Epoch {epoch}", leave=False):
            batch = batch.to(device)
            model.beta = beta

            loss_result = model(batch)

            if train:
                optimizer.zero_grad()
                loss_result["loss"].backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            for k in totals:
                totals[k] += loss_result[k].item()

    n = len(loader)
    return {k: v / n for k, v in totals.items()}


def main():
    args_dict = {
        "processed_dir": "./data/processed",
        "save_dir": "./checkpoints",
        "epochs": 100,
        "batch_size": 128,
        "lr": 1e-3,
        "n_cycles": 4,
        "max_beta": 0.5,
        "warmup_epochs": 10,
        "save_every": 10,
        "num_workers": 0,
        "max_samples": 100_000,  # None이면 전체 데이터 사용
    }
    args = SimpleNamespace(**args_dict)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    save_dir = Path(args.save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    print("Loading datasets ...")

    train_set = QMugsDataset(args.processed_dir, split="train", max_samples=args.max_samples)
    val_set   = QMugsDataset(args.processed_dir, split="val",   max_samples=args.max_samples // 5 if args.max_samples else None)

    train_loader = DataLoader(train_set, batch_size=args.batch_size, shuffle=False, num_workers=args.num_workers)
    val_loader   = DataLoader(val_set,   batch_size=args.batch_size, shuffle=False, num_workers=args.num_workers)
    print(f"  Train: {len(train_set):,}  Val: {len(val_set):,}")

    model = MoleculeCVAE().to(device)
    optimizer = AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=args.epochs, eta_min=1e-5)

    total_steps = args.epochs * len(train_loader)
    warmup_steps = args.warmup_epochs * len(train_loader)
    step = 0
    best_val = float("inf")

    print(f"  Warmup: {args.warmup_epochs} epochs ({warmup_steps:,} steps), "
          f"then {args.n_cycles} cosine cycles")
    print(f"\n{'Epoch':>6} {'Beta':>6} {'T-loss':>8} {'T-recon':>8} "
          f"{'T-exist':>8} {'T-type':>8} {'T-kl':>8} "
          f"{'V-loss':>8} {'V-recon':>8}  Time")
    print("-" * 95)

    for epoch in range(1, args.epochs + 1):
        t0 = time.time()

        beta = cyclical_beta(step, total_steps, args.n_cycles, args.max_beta,
                             warmup_steps=warmup_steps)

        train_result = run_epoch(model, train_loader, optimizer, beta, device, train=True, epoch=epoch)
        val_result   = run_epoch(model, val_loader,   optimizer, beta, device, train=False, epoch=epoch)

        step += len(train_loader)
        scheduler.step()

        time_elapsed = time.time() - t0
        print(
            f"{epoch:>6} {beta:>6.3f} "
            f"{train_result['loss']:>8.4f} {train_result['reconstruction_loss']:>8.4f} "
            f"{train_result['exist_loss']:>8.4f} {train_result['type_loss']:>8.4f} "
            f"{train_result['kl_loss']:>8.4f} "
            f"{val_result['loss']:>8.4f} {val_result['reconstruction_loss']:>8.4f} "
            f" {time_elapsed:>5.1f}s"
        )

        if val_result["loss"] < best_val:
            best_val = val_result["loss"]
            torch.save(model.state_dict(), save_dir / "best.pt")

        if args.save_every != 0 and epoch % args.save_every == 0:
            torch.save({
                "epoch": epoch,
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "best_val": best_val
            }, save_dir / f"ckpt_epoch{epoch:04d}.pt")

    print(f"\ntrain finished! Best val loss: {best_val:.4f}")

main()